# Regime-Aware MoE ES — Multi-Horizon Optuna

RegimeAwareMoE with rasterized short stream + macro market-state gating.
USA session (CT 08:30–15:45). **Target horizon is an Optuna hyperparameter** (1h–8h).

Architecture: RegimeRouter inspects x_market → expert weights, each Expert =
MarketGatingUnit → CMDMamba backbone, weighted fusion → unified head.

Uses `backtest_from_predictions` with `TradingMode.CONTINUOUS`.

## 1. Environment Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

In [ ]:
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    %cd CTAFlow/
    !git pull
    %cd ..

    sys.path.insert(0, "/content/drive/MyDrive/CTAEnv/CTAFlow/")
    sys.path.insert(1, "/content/drive/MyDrive/CTAEnv/SierraPy")
    !pip install -e CTAFlow -q
    !pip install optuna yfinance pandas-datareader -q
else:
    print("Running locally - ensure CTAFlow is installed")

## 2. Imports

In [ ]:
import json, warnings, joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.metrics import confusion_matrix, classification_report

from CTAFlow.data.raw_formatting.intraday_manager import read_exported_df
from CTAFlow.data.ext.macro_client import MacroClient
from CTAFlow.models.prep.intraday_continuous import (
    FinancialsIntradayPrep,
    SessionSpec,
)
from CTAFlow.data.datasets.continuous import (
    FinMambaContinuousDataset,
    collate_finmamba_continuous,
)
from CTAFlow.models.deep_learning.multi_branch.regime_moe import (
    RegimeAwareMoE,
    RegimeMoEConfig,
)
from CTAFlow.models.deep_learning.multi_branch.cmd_mamba import CMDMambaConfig
from CTAFlow.models.deep_learning.training.loss import ProfitWeightedCE
from CTAFlow.models.deep_learning.training.backtest import (
    backtest_from_predictions,
    TradingMode,
)

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}  |  torch {torch.__version__}  |  optuna {optuna.__version__}")

## 3. Configuration

In [ ]:
# --- Ticker / bar ---
TICKER = "ES"
BAR_RULE = "15min"
BAR_MIN = 15

# --- Session (USA, CT) ---
SESSION_START = "08:30"
SESSION_END   = "15:45"

# --- Multi-horizon targets (up to 8h) ---
# steps_60m covers the longest horizon: 8h / 15min = 32 steps
MAX_TARGET_STEPS = 32
NUM_CLASSES = 3
CLF_PERCENTILES = (30, 70)

# Candidate horizons for Optuna search
HORIZONS = {
    "1h":  60 // BAR_MIN,    # 4 steps
    "2h":  120 // BAR_MIN,   # 8 steps
    "4h":  240 // BAR_MIN,   # 16 steps
    "8h":  480 // BAR_MIN,   # 32 steps
}
HORIZON_NAMES = list(HORIZONS.keys())

# --- Sequence lengths ---
LONG_LOOKBACK = 192
SHORT_RANGE = (24, 96)
LONG_PATCH_OPTIONS = [2, 4, 8]

# --- Sampling stride ---
TRAIN_STRIDE = 4
VAL_STRIDE = 1

# --- Split / quality ---
VAL_SPLIT = 0.2
MAX_MISS_ROWS = 6
MAX_MISS_RATIO = 0.20

# --- Optuna ---
N_TRIALS = 30
TRIAL_EPOCHS = 20
FINAL_EPOCHS = 40
PATIENCE = 10

# --- Backtest costs (bps, continuous mode) ---
TRANSACTION_COST_BPS = 2.5
SLIPPAGE_BPS = 1.5

# --- Paths ---
if IN_COLAB:
    DATA_DIR = Path(f"/content/drive/MyDrive/features/{TICKER}")
else:
    DATA_DIR = Path(f"/workspace/model_data/{TICKER}")
DATA_PATH = DATA_DIR / "intraday.csv"
RASTER_PATH = DATA_DIR / f"{TICKER}_rasterized.npz"
RESULTS_PATH = DATA_DIR / "results" / "regime_moe_es_optuna"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# --- Download from S3 if not in Colab and data missing ---
if not IN_COLAB and (not DATA_PATH.exists() or not RASTER_PATH.exists()):
    import os
    from CTAFlow.data.storage.aws_client import AWSClient, S3Config

    _bucket = os.getenv("CTAFLOW_S3_BUCKET", "ctaflow-data")
    _region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    _endpoint = os.getenv("AWS_ENDPOINT_URL")

    print(f"Data missing locally - downloading {TICKER} from S3 ({_bucket})...")
    _s3 = AWSClient(S3Config(
        bucket_name=_bucket,
        region=_region,
        endpoint_url=_endpoint,
    ))
    _downloaded = _s3.download_ticker_data(
        ticker=TICKER,
        local_dir=str(DATA_DIR),
        data_types=["intraday", "rasterized"],
    )
    if not _downloaded:
        _s3.download_directory(
            s3_prefix=f"workspace/model_data/{TICKER}/",
            local_dir=str(DATA_DIR),
        )
    print(f"  Downloaded: {list(_downloaded.keys()) if _downloaded else 'directory'}")

assert DATA_PATH.exists(), f"Missing {DATA_PATH}"
assert RASTER_PATH.exists(), f"Missing {RASTER_PATH}"

print(f"data:     {DATA_PATH}")
print(f"raster:   {RASTER_PATH}")
print(f"results:  {RESULTS_PATH}")
print(f"horizons: {HORIZONS}")
print(f"stride:   train={TRAIN_STRIDE}  val={VAL_STRIDE}")

## 4. Load Data, Fetch Macro, Prepare Features

1. Load raw ticks and resample to 15-min bars
2. Fetch macro context (SPX, VIX, yields, sector ETFs) via `MacroClient`
3. Build asset + macro features via `FinancialsIntradayPrep` with 1-day lag
4. Create multi-step targets covering 15min–8h horizons

In [ ]:
def resample_bars(df, rule):
    agg = {"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"}
    for c in df.columns:
        if c not in agg and np.issubdtype(df[c].dtype, np.number):
            agg[c] = "last"
    return df.resample(rule).agg(agg).dropna(subset=["Open", "High", "Low", "Close"]).sort_index()


# ---- load raw data and resample ----
raw = read_exported_df(str(DATA_PATH))
bars = resample_bars(raw, BAR_RULE)
print(f"bars: {len(bars)}  |  {bars.index.min()} -> {bars.index.max()}")

# ---- load raster metadata ----
rz = np.load(RASTER_PATH, allow_pickle=False)
rdata, ridx = rz["data"], pd.to_datetime(rz["idx"])
RBINS = int(rdata.shape[2])
print(f"raster: {rdata.shape}  |  {ridx.min()} -> {ridx.max()}")

# ---- fetch macro context (1-day lagged in FinancialsIntradayPrep) ----
macro_start = bars.index.min() - pd.Timedelta(days=60)
macro_end = bars.index.max()
print(f"\nFetching macro context: {macro_start.date()} -> {macro_end.date()}")
macro_client = MacroClient()
macro_raw = macro_client.get_macro_context(
    start_date=macro_start, end_date=macro_end,
)
print(f"macro_raw: {macro_raw.shape}  |  cols: {list(macro_raw.columns)}")

# ---- prep with USA session + macro features ----
prep = FinancialsIntradayPrep(
    sessions=[
        SessionSpec("LONDON", "02:00", "11:00"),
        SessionSpec("USA", SESSION_START, SESSION_END),
    ],
    bar_minutes=BAR_MIN,
    macro_df=macro_raw,
)

print(f"\nmacro feature cols ({len(prep.macro_feature_cols)}): {prep.macro_feature_cols[:10]}...")

df, train_mask, tcols = prep.prepare(
    bars,
    steps_60m=MAX_TARGET_STEPS,
    keep_only_active=False,
    add_daily=True,
    add_overnight=True,
    add_deseas=True,
    add_time_features=True,
    add_resample_precalc=True,
    resample_rules=("30min", "60min"),
    rolling_days_deseas=252,
    refit_interval=10,
    use_legacy_deseas=False,
    apply_scaling=True,
    scale_to_basis_points=True,
)

# ---- filter to USA session only ----
active = df["is_usa"].astype(bool)
df = df[active].copy()
train_mask = train_mask.loc[df.index]

print(f"\nraw target cols from prepare(): {tcols}")
print(f"USA session rows: {len(df)}")

## 5. Multi-Horizon Classification

Pre-compute class labels for each candidate horizon (1h, 2h, 4h, 8h).
All are stored in the DataFrame so the dataset only needs to be built once.
Optuna picks the horizon per trial.

In [ ]:
# ---- select asset feature columns (exclude macro, targets, OHLCV) ----
macro_set = set(prep.macro_feature_cols)
exclude = set(
    tcols
    + ["Open", "High", "Low", "Close", "Volume",
       "session_code", "is_active", "is_usa", "is_london"]
) | macro_set
FEAT = [
    c for c in df.columns
    if c not in exclude
    and not c.startswith("target_")
    and np.issubdtype(df[c].dtype, np.number)
    and pd.notna(df[c].std())
    and df[c].std() > 0
]
MARKET_FEAT = [c for c in prep.macro_feature_cols if c in df.columns]

# ---- classify each horizon ----
horizon_thresholds = {}  # {name: (q_low, q_high)}
all_target_cols = []     # ordered: [class_1h, return_1h, class_2h, return_2h, ...]

for name, steps in HORIZONS.items():
    raw_col = f"y_fwd_{steps}"
    cls_col = f"target_class_{name}"
    ret_col = f"target_return_{name}"

    tr_returns = df.loc[train_mask, raw_col].dropna()
    q_low, q_high = np.percentile(tr_returns, CLF_PERCENTILES)
    horizon_thresholds[name] = (float(q_low), float(q_high))

    df[cls_col] = df[raw_col].apply(
        lambda x: -1 if pd.isna(x) else (0 if x <= q_low else (1 if x <= q_high else 2))
    ).astype(np.int64)
    df[ret_col] = df[raw_col].astype(np.float32)

    all_target_cols.extend([cls_col, ret_col])
    print(f"{name:>3s} ({steps:2d} steps): q_low={q_low:+.6f}  q_high={q_high:+.6f}")

# ---- drop rows missing the longest horizon target ----
longest_cls = f"target_class_{HORIZON_NAMES[-1]}"
valid = (df[longest_cls] >= 0)
for name in HORIZON_NAMES:
    valid &= df[f"target_return_{name}"].notna()
df = df[valid].copy()

print(f"\nasset features:  {len(FEAT)}")
print(f"market features: {len(MARKET_FEAT)}")
print(f"target cols ({len(all_target_cols)}): {all_target_cols}")
print(f"samples after filter: {len(df)}")

# ---- plot class distributions per horizon ----
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(4 * len(HORIZONS), 4))
for ax, name in zip(axes, HORIZON_NAMES):
    vc = df[f"target_class_{name}"].value_counts().sort_index()
    ax.bar(["Short", "Flat", "Long"], vc.values,
           color=["#e74c3c", "#95a5a6", "#27ae60"])
    ax.set_title(f"{name} horizon")
plt.suptitle("Class Distributions by Horizon", fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Alignment Diagnostic

In [ ]:
from CTAFlow.data.datasets.continuous import (
    _normalize_ts_index, _build_session_index, _resample_frame,
)

work = df.copy()
work.index = _normalize_ts_index(work.index)
work = work.sort_index()
work = _resample_frame(work, BAR_RULE)

rz = np.load(str(RASTER_PATH), allow_pickle=False)
raster_idx = _normalize_ts_index(pd.to_datetime(rz["idx"]))

print("=== Index timezone info ===")
print(f"  work.index.tz  : {work.index.tz}")
print(f"  raster_idx.tz  : {raster_idx.tz}")

print(f"\n=== Date ranges ===")
print(f"  work   : {work.index.min()} -> {work.index.max()}  ({len(work)} rows)")
print(f"  raster : {raster_idx.min()} -> {raster_idx.max()}  ({len(raster_idx)} rows)")

work_days = set(work.index.normalize())
raster_days = set(raster_idx.normalize())
common_days = sorted(work_days & raster_days)
print(f"\n=== Day overlap ===")
print(f"  work days    : {len(work_days)}")
print(f"  raster days  : {len(raster_days)}")
print(f"  common days  : {len(common_days)}")

if common_days:
    test_day = pd.Timestamp(common_days[len(common_days) // 2])
    expected = _build_session_index(test_day, SESSION_START, SESSION_END, BAR_RULE)
    print(f"\n=== Sample day: {test_day.date()} ===")
    print(f"  expected timestamps ({len(expected)}): {expected[0]} -> {expected[-1]}")
    work_hits = expected.isin(work.index).sum()
    raster_hits = expected.isin(raster_idx).sum()
    print(f"  expected in work   : {work_hits}/{len(expected)}")
    print(f"  expected in raster : {raster_hits}/{len(expected)}")

    # Per-day filter summary
    n_ok, n_drop = 0, 0
    for day in common_days:
        exp = _build_session_index(day, SESSION_START, SESSION_END, BAR_RULE)
        if len(exp) == 0:
            n_drop += 1
            continue
        ridx_pos = raster_idx.get_indexer(exp)
        n_miss = int((ridx_pos < 0).sum())
        miss_limit = max(MAX_MISS_ROWS, int(np.floor(MAX_MISS_RATIO * len(exp))))
        if not np.any(ridx_pos >= 0) or n_miss > miss_limit:
            n_drop += 1
        else:
            n_ok += 1
    print(f"\n=== Per-day filter: passed={n_ok}  dropped={n_drop} ===")

## 7. Dataset and Loaders

Build `FinMambaContinuousDataset` with all horizon targets packed into one tensor.
Each batch returns `y` of shape `[B, 8]` = `[class_1h, ret_1h, class_2h, ret_2h, ...]`.
Optuna trials select the horizon-specific class/return pair at training time.

In [ ]:
ds_kwargs = dict(
    df=df,
    raster_npz_path=str(RASTER_PATH),
    feature_cols=FEAT,
    target_cols=all_target_cols,
    market_feature_cols=MARKET_FEAT,
    long_lookback=LONG_LOOKBACK,
    short_len_range=SHORT_RANGE,
    resample_rule=BAR_RULE,
    session_start=SESSION_START,
    session_end=SESSION_END,
    sample_mode="usa",
    allow_overlap=False,
    max_missing_rows_per_day=MAX_MISS_ROWS,
    max_missing_ratio_per_day=MAX_MISS_RATIO,
    return_meta=True,
)

train_base = FinMambaContinuousDataset(
    **ds_kwargs, random_short_len=True, sample_stride=TRAIN_STRIDE,
)
eval_base = FinMambaContinuousDataset(
    **ds_kwargs, random_short_len=False, sample_stride=VAL_STRIDE,
)

# time-ordered split
sample_ts = pd.DatetimeIndex(eval_base.df.index[eval_base.sample_pos])
days = pd.Index(sample_ts.normalize().unique()).sort_values()
split_day = days[int(len(days) * (1 - VAL_SPLIT))]

va_idx = np.flatnonzero(sample_ts >= split_day)

train_ts = pd.DatetimeIndex(train_base.df.index[train_base.sample_pos])
tr_idx_strided = np.flatnonzero(train_ts < split_day)
train_ds = Subset(train_base, tr_idx_strided.tolist())
val_ds   = Subset(eval_base, va_idx.tolist())


def make_loaders(bs):
    pin = torch.cuda.is_available()
    tr = DataLoader(train_ds, batch_size=bs, shuffle=True,
                    num_workers=0, pin_memory=pin,
                    collate_fn=collate_finmamba_continuous)
    va = DataLoader(val_ds, batch_size=bs, shuffle=False,
                    num_workers=0, pin_memory=pin,
                    collate_fn=collate_finmamba_continuous)
    return tr, va


# --- horizon index mapping ---
# y tensor layout: [class_1h, ret_1h, class_2h, ret_2h, class_4h, ret_4h, class_8h, ret_8h]
HORIZON_CLS_IDX = {name: i * 2     for i, name in enumerate(HORIZON_NAMES)}
HORIZON_RET_IDX = {name: i * 2 + 1 for i, name in enumerate(HORIZON_NAMES)}

# quick sanity check
tr_loader, _ = make_loaders(16)
batch = next(iter(tr_loader))
has_meta = len(batch) == 7
if has_meta:
    xS, xL, xM, tS, tL, y, meta = batch
else:
    xS, xL, xM, tS, tL, y = batch

print(f"split:  {split_day.date()}")
print(f"train: {len(train_ds)} (stride={TRAIN_STRIDE})  |  val: {len(val_ds)} (stride={VAL_STRIDE})")
print(f"xS: {xS.shape}  xL: {xL.shape}  xM: {xM.shape}  tS: {tS.shape}  tL: {tL.shape}  y: {y.shape}")
print(f"raster_bins: {RBINS}  asset_feats: {len(FEAT)}  market_feats: {len(MARKET_FEAT)}")
print(f"\ny columns: {all_target_cols}")
print(f"horizon index map (class): {HORIZON_CLS_IDX}")
print(f"horizon index map (return): {HORIZON_RET_IDX}")

## 8. Model, Training Helpers, and Optuna Objective

**Model**: `RegimeAwareMoE` = `RegimeRouter` + N x (`MarketGatingUnit` → `CMDMamba`) + unified head.

**Key differences from FinMambaCMD**:
- **Aux loss** from the router (load-balancing) is added to the task loss
- **num_experts**, **top_k**, **router_temperature** are hyperparameters
- **Target horizon** (1h–8h) is a categorical hyperparameter

**Scoring**: `score = pnl * (acc - baseline)` when margin >= 2%, else `pnl * (acc / baseline)`

In [ ]:
TIME_DIM = len(eval_base.time_feature_cols)


# ---------------------------------------------------------------------------
# Model builder
# ---------------------------------------------------------------------------
def build_model(p):
    cmd_cfg = CMDMambaConfig(
        base_bar_minutes=BAR_MIN,
        long_patch=int(p["long_patch"]),
        long_stride=int(p["long_patch"]),
        n_long_layers=int(p["n_long_layers"]),
        n_short_layers=int(p["n_short_layers"]),
        raster_channels=4,
        raster_bins=RBINS,
        d_model=int(p["d_model"]),
        dropout=float(p["dropout"]),
        d_state_long=int(p["d_state_long"]),
        d_state_short=int(p["d_state_short"]),
        task="classification",
        out_dim=NUM_CLASSES,
    )
    moe_cfg = RegimeMoEConfig(
        num_experts=int(p["num_experts"]),
        router_hidden_dim=int(p["router_hidden_dim"]),
        router_dropout=float(p["dropout"]),
        router_temperature=float(p["router_temperature"]),
        top_k=int(p["top_k"]),
        aux_loss_coeff=float(p["aux_loss_coeff"]),
    )
    return RegimeAwareMoE(
        asset_dim=len(FEAT),
        market_dim=len(MARKET_FEAT),
        cmd_config=cmd_cfg,
        moe_config=moe_cfg,
        time_feat_dim=TIME_DIM,
        classification=True,
        num_classes=NUM_CLASSES,
    ).to(device)


# ---------------------------------------------------------------------------
# Scoring: accuracy-weighted PnL
#   margin >= 2%  -> weight by (acc - baseline)  (reward clear edge)
#   margin < 2%   -> weight by (acc / baseline)  (smooth near-baseline)
# ---------------------------------------------------------------------------
def compute_accuracy_weighted_pnl(accuracy, pnl, baseline):
    margin = accuracy - baseline
    weight = margin if margin >= 2.0 else (accuracy / baseline)
    return float(pnl * weight)


# ---------------------------------------------------------------------------
# Batch unpacking with horizon selection
# ---------------------------------------------------------------------------
def unpack(batch, horizon_name):
    """Unpack batch and extract the class/return pair for the given horizon."""
    if len(batch) == 7:
        xS, xL, xM, tS, tL, y, meta = batch
    else:
        xS, xL, xM, tS, tL, y = batch
        meta = None

    cls_idx = HORIZON_CLS_IDX[horizon_name]
    ret_idx = HORIZON_RET_IDX[horizon_name]

    return (
        xS.to(device), xL.to(device), xM.to(device),
        tS.to(device), tL.to(device),
        y[:, cls_idx].long().to(device),    # class label for this horizon
        y[:, ret_idx].float().to(device),   # raw return for this horizon
        meta,
    )


# ---------------------------------------------------------------------------
# Train / evaluate one epoch
# ---------------------------------------------------------------------------
def train_epoch(model, loader, criterion, optimizer, horizon_name):
    model.train()
    total_loss = total_correct = total_n = 0
    total_pnl = total_aux = 0.0

    for batch in loader:
        xS, xL, xM, tS, tL, yc, yr, _ = unpack(batch, horizon_name)
        optimizer.zero_grad()

        # return_features=True to get aux_loss
        out, extras = model(
            x_short=xS, x_long=xL, x_market=xM,
            t_short=tS, t_long=tL,
            return_features=True,
        )

        task_loss = (
            criterion(out, yc, returns=yr)
            if isinstance(criterion, ProfitWeightedCE)
            else criterion(out, yc)
        )
        aux = model.aux_loss(extras)
        loss = task_loss + aux

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        with torch.no_grad():
            probs = torch.softmax(out, dim=-1)
            preds = probs.argmax(dim=-1)
            position = probs[:, 2] - probs[:, 0]
            total_loss += loss.item() * yc.size(0)
            total_correct += (preds == yc).sum().item()
            total_pnl += (position * yr).sum().item()
            total_aux += aux.item() * yc.size(0)
            total_n += yc.size(0)

    n = max(1, total_n)
    return total_loss / n, 100.0 * total_correct / n, total_pnl / n, total_aux / n


def evaluate(model, loader, criterion, horizon_name, ret_arrays=False):
    model.eval()
    total_loss = total_correct = total_n = 0
    total_pnl = 0.0
    all_logits, all_labels, all_returns, all_dates = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            xS, xL, xM, tS, tL, yc, yr, meta = unpack(batch, horizon_name)
            out = model(
                x_short=xS, x_long=xL, x_market=xM,
                t_short=tS, t_long=tL,
            )

            loss = (
                criterion(out, yc, returns=yr)
                if isinstance(criterion, ProfitWeightedCE)
                else criterion(out, yc)
            )
            probs = torch.softmax(out, dim=-1)
            preds = probs.argmax(dim=-1)
            position = probs[:, 2] - probs[:, 0]

            total_loss += loss.item() * yc.size(0)
            total_correct += (preds == yc).sum().item()
            total_pnl += (position * yr).sum().item()
            total_n += yc.size(0)

            if ret_arrays:
                all_logits.append(out.cpu().numpy())
                all_labels.append(yc.cpu().numpy())
                all_returns.append(yr.cpu().numpy())
                if meta is not None:
                    all_dates.extend(
                        m.get("timestamp") if isinstance(m, dict) else None
                        for m in meta
                    )

    n = max(1, total_n)
    avg_loss = total_loss / n
    acc = 100.0 * total_correct / n
    avg_pnl = total_pnl / n

    if not ret_arrays:
        return avg_loss, acc, avg_pnl

    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    returns = np.concatenate(all_returns)
    dates = pd.to_datetime(np.array(all_dates)) if all_dates else None
    _, cnt = np.unique(labels, return_counts=True)
    baseline = 100.0 * cnt.max() / max(1, len(labels))

    return avg_loss, acc, avg_pnl, baseline, logits, labels, returns, dates


# ---------------------------------------------------------------------------
# Optuna objective
# ---------------------------------------------------------------------------
def objective(trial):
    p = {
        # Target horizon
        "target_horizon":   trial.suggest_categorical("target_horizon", HORIZON_NAMES),
        # Batch / backbone
        "batch_size":       trial.suggest_categorical("batch_size", [16, 32, 64]),
        "d_model":          trial.suggest_categorical("d_model", [64, 128, 256]),
        "n_long_layers":    trial.suggest_int("n_long_layers", 2, 4),
        "n_short_layers":   trial.suggest_int("n_short_layers", 1, 3),
        "d_state_long":     trial.suggest_categorical("d_state_long", [32, 64, 128]),
        "d_state_short":    trial.suggest_categorical("d_state_short", [8, 16, 32]),
        "long_patch":       trial.suggest_categorical("long_patch", LONG_PATCH_OPTIONS),
        "dropout":          trial.suggest_float("dropout", 0.1, 0.4),
        # MoE architecture
        "num_experts":      trial.suggest_categorical("num_experts", [2, 3, 4]),
        "router_hidden_dim": trial.suggest_categorical("router_hidden_dim", [32, 64, 128]),
        "router_temperature": trial.suggest_float("router_temperature", 0.5, 2.0),
        "top_k":            trial.suggest_categorical("top_k", [0, 2]),  # 0=soft, 2=sparse
        "aux_loss_coeff":   trial.suggest_float("aux_loss_coeff", 0.001, 0.1, log=True),
        # Optimiser
        "learning_rate":    trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True),
        "weight_decay":     trial.suggest_float("weight_decay", 0.01, 0.1, log=True),
        # Loss
        "profit_scale":     trial.suggest_float("profit_scale", 1.0, 20.0, log=True),
        "direction_penalty": trial.suggest_float("direction_penalty", 0.5, 2.0),
        "min_weight":       trial.suggest_float("min_weight", 0.5, 1.0),
        "max_weight":       trial.suggest_float("max_weight", 1.0, 5.0),
    }

    horizon_name = p["target_horizon"]
    tr, va = make_loaders(int(p["batch_size"]))
    model = build_model(p)
    opt = optim.AdamW(
        model.parameters(), lr=p["learning_rate"], weight_decay=p["weight_decay"],
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TRIAL_EPOCHS)

    ce  = nn.CrossEntropyLoss()
    pce = ProfitWeightedCE(
        profit_scale=p["profit_scale"], min_weight=p["min_weight"],
        max_weight=p["max_weight"], direction_penalty=p["direction_penalty"],
    )

    best_score = -1e9
    patience_counter = 0
    best_metrics = {}

    for epoch in range(TRIAL_EPOCHS):
        crit = ce if epoch < 3 else pce
        _, tr_acc, tr_pnl, tr_aux = train_epoch(model, tr, crit, opt, horizon_name)
        val_loss, val_acc, val_pnl, baseline, logits, labels, returns, dates = evaluate(
            model, va, crit, horizon_name, ret_arrays=True,
        )
        scheduler.step()

        score = compute_accuracy_weighted_pnl(val_acc, val_pnl, baseline)
        acc_margin = val_acc - baseline

        if score > best_score:
            best_score = score
            patience_counter = 0

            bt_result = backtest_from_predictions(
                predictions=logits, returns=returns, dates=dates,
                task="classification", long_class=2, short_class=0,
                transaction_cost_bps=TRANSACTION_COST_BPS,
                slippage_bps=SLIPPAGE_BPS,
                mode=TradingMode.CONTINUOUS,
            )

            tracker = model.get_last_tracker()
            rw = tracker.get("router_weights_mean", torch.zeros(1))

            best_metrics = {
                "final_acc": val_acc,
                "final_loss": val_loss,
                "final_pnl": val_pnl,
                "max_class_pct": baseline,
                "acc_margin": acc_margin,
                "acc_weighted_score": score,
                "sharpe_like": bt_result.summary["sharpe_like"],
                "max_drawdown": bt_result.summary["max_drawdown"],
                "n_trades": bt_result.trade_stats.get("n_trades", 0),
                "win_rate": bt_result.trade_stats.get("win_rate", 0.0),
                "profit_factor": bt_result.trade_stats.get("profit_factor", 0.0),
                "dominant_expert": int(tracker.get("dominant_expert", -1)),
                "router_weights": [float(x) for x in rw],
                "aux_loss_last": float(tracker.get("aux_loss", 0)),
            }
        else:
            patience_counter += 1

        print(
            f"  [MoE-{horizon_name}] E{epoch+1:02} | Acc: {val_acc:.1f}% "
            f"(base: {baseline:.1f}%, +{acc_margin:.1f}) | "
            f"PnL: {val_pnl:.6f} | Aux: {tr_aux:.4f} | Score: {score:.4f}"
        )

        trial.report(score, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        if patience_counter >= PATIENCE:
            break

    for k, v in best_metrics.items():
        if isinstance(v, (list, tuple)):
            trial.set_user_attr(k, v)
        else:
            trial.set_user_attr(k, float(v))
    trial.set_user_attr("n_params", sum(pp.numel() for pp in model.parameters()))

    return float(best_score)

## 9. Run Optimisation and Analyse Results

In [ ]:
STUDY_NAME = f"regime_moe_{TICKER}_macro_clf"

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="maximize",
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# --- Best trial summary ---
print("\n" + "=" * 60)
print("BEST TRIAL SUMMARY")
print("=" * 60)

best = study.best_trial
acc = best.user_attrs.get("final_acc", 0)
max_class = best.user_attrs.get("max_class_pct", 33.33)
acc_margin = best.user_attrs.get("acc_margin", acc - max_class)

print(f"\nBest Horizon: {best.params['target_horizon']}")
print(f"Accuracy-Weighted PnL Score: {best.value:.4f}")
print(f"\n--- Accuracy ---")
print(f"  Accuracy: {acc:.2f}%")
print(f"  Baseline (max class): {max_class:.2f}%")
print(f"  Margin: {acc_margin:+.2f}%")
print(f"\n--- Trading (continuous mode) ---")
print(f"  Raw PnL: {best.user_attrs.get('final_pnl', 0):.6f}")
print(f"  Sharpe-like: {best.user_attrs.get('sharpe_like', 0):.3f}")
print(f"  Win Rate: {best.user_attrs.get('win_rate', 0):.2%}")
print(f"  Profit Factor: {best.user_attrs.get('profit_factor', 0):.2f}")
print(f"  # Trades: {best.user_attrs.get('n_trades', 0):.0f}")
print(f"  Max Drawdown: {best.user_attrs.get('max_drawdown', 0):.6f}")
print(f"\n--- MoE Routing ---")
print(f"  Experts: {best.params['num_experts']}")
print(f"  Top-K: {best.params['top_k']} ({'soft' if best.params['top_k'] == 0 else 'sparse'})")
print(f"  Dominant Expert: {best.user_attrs.get('dominant_expert', '?')}")
print(f"  Router Weights: {best.user_attrs.get('router_weights', [])}")
print(f"  Aux Loss (last): {best.user_attrs.get('aux_loss_last', 0):.4f}")
print(f"\n--- Model ---")
print(f"  Parameters: {best.user_attrs.get('n_params', 0):,}")
for k in ["d_model", "long_patch", "n_long_layers", "n_short_layers",
          "d_state_long", "d_state_short", "learning_rate", "weight_decay",
          "router_temperature", "aux_loss_coeff"]:
    if k in best.params:
        print(f"  {k}: {best.params[k]}")

# ---- trial analysis plots ----
tdf = study.trials_dataframe()
comp = tdf[tdf["state"] == "COMPLETE"].copy()
print(f"\nCompleted trials: {len(comp)}/{N_TRIALS}")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].hist(comp["value"], bins=20, alpha=0.7, edgecolor="black")
axes[0, 0].axvline(best.value, c="r", ls="--", label=f"best={best.value:.4f}")
axes[0, 0].legend()
axes[0, 0].set_title("Score Distribution")

axes[0, 1].plot(comp["number"], comp["value"], "o-", alpha=0.7)
axes[0, 1].set_title("Optimisation History")

# Score by horizon
if "params_target_horizon" in comp.columns:
    for h in HORIZON_NAMES:
        mask = comp["params_target_horizon"] == h
        axes[0, 2].scatter(
            comp.loc[mask, "number"], comp.loc[mask, "value"],
            label=h, s=70, alpha=0.8,
        )
    axes[0, 2].legend()
axes[0, 2].set_title("Score by Target Horizon")

for ax, col, title in [
    (axes[1, 0], "params_num_experts",        "num_experts vs Score"),
    (axes[1, 1], "params_d_model",             "d_model vs Score"),
]:
    if col in comp.columns:
        ax.scatter(comp[col], comp["value"], alpha=0.7, s=70)
    ax.set_title(title)

if "params_router_temperature" in comp.columns:
    axes[1, 2].scatter(comp["params_router_temperature"], comp["value"], alpha=0.7, s=70)
axes[1, 2].set_title("Router Temperature vs Score")

for ax in axes.ravel():
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f"Regime-Aware MoE Optuna \u2014 {TICKER} USA (Multi-Horizon, Continuous)",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{STUDY_NAME}_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- parameter importance ----
imp = optuna.importance.get_param_importances(study)
idf = pd.DataFrame({"param": list(imp.keys()), "importance": list(imp.values())})

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=idf, x="importance", y="param", ax=ax)
ax.grid(True, alpha=0.3)
ax.set_title("Hyperparameter Importance (includes target_horizon)")
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{STUDY_NAME}_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Final Training, Continuous Backtest, Expert Analysis, and Save

Retrain with best params using the optimal horizon. Includes:
- Backtest with `TradingMode.CONTINUOUS`
- Per-expert routing weight analysis across the validation set
- Expert specialisation diagnostics

In [ ]:
p = study.best_params.copy()
best_horizon = p["target_horizon"]
print(f"Best horizon: {best_horizon} ({HORIZONS[best_horizon]} steps x {BAR_MIN}min)")

tr, va = make_loaders(int(p["batch_size"]))
model = build_model(p)
opt = optim.AdamW(
    model.parameters(), lr=p["learning_rate"], weight_decay=p["weight_decay"],
)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2)
crit = ProfitWeightedCE(
    profit_scale=p["profit_scale"], min_weight=p["min_weight"],
    max_weight=p["max_weight"], direction_penalty=p["direction_penalty"],
)

print(f"Training final RegimeAwareMoE model...")
print(f"Parameters: {sum(pp.numel() for pp in model.parameters()):,}")
print(f"Experts: {p['num_experts']}  Top-K: {p['top_k']}  Temperature: {p['router_temperature']:.2f}")
print(f"Train: {len(train_ds)} (stride={TRAIN_STRIDE})  |  Val: {len(val_ds)} (stride={VAL_STRIDE})")

hist = {
    "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
    "val_pnl": [], "val_score": [], "aux_loss": [],
}
best_score = -1e9
best_state = None
patience_counter = 0

for epoch in range(FINAL_EPOCHS):
    tl, ta, tp, aux = train_epoch(model, tr, crit, opt, best_horizon)
    vl, va_acc, vp, baseline, *_ = evaluate(
        model, va, crit, best_horizon, ret_arrays=True,
    )
    sc = compute_accuracy_weighted_pnl(va_acc, vp, baseline)
    scheduler.step()

    hist["train_loss"].append(tl)
    hist["val_loss"].append(vl)
    hist["train_acc"].append(ta)
    hist["val_acc"].append(va_acc)
    hist["val_pnl"].append(vp)
    hist["val_score"].append(sc)
    hist["aux_loss"].append(aux)

    if sc > best_score:
        best_score = sc
        best_state = {
            k: v.detach().cpu().clone() for k, v in model.state_dict().items()
        }
        patience_counter = 0
    else:
        patience_counter += 1

    print(
        f"E{epoch+1:02d}  loss={tl:.4f}/{vl:.4f}  "
        f"acc={va_acc:.2f}% (base={baseline:.1f}%, +{va_acc-baseline:.1f})  "
        f"pnl={vp:.6f}  aux={aux:.4f}  score={sc:.4f}"
    )
    if patience_counter >= PATIENCE:
        print(f"  -> early stop at epoch {epoch+1}")
        break

# ---- reload best checkpoint ----
if best_state is not None:
    model.load_state_dict(best_state)

# ---- final evaluation + continuous backtest ----
vl, va_acc, vp, baseline, logits, ytrue, rets, dates = evaluate(
    model, va, crit, best_horizon, ret_arrays=True,
)

bt = backtest_from_predictions(
    predictions=logits, returns=rets, dates=dates,
    task="classification", long_class=2, short_class=0,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    slippage_bps=SLIPPAGE_BPS,
    mode=TradingMode.CONTINUOUS,
)

print(f"\n{'=' * 60}")
print(f"FINAL BACKTEST RESULTS (Continuous Mode, Horizon: {best_horizon})")
print(f"{'=' * 60}")
print(f"\nval loss={vl:.4f}  acc={va_acc:.2f}%  baseline={baseline:.2f}%  margin={va_acc - baseline:+.2f}%")
print(f"\nNet PnL: {bt.summary['net_pnl_sum']:.4f}")
print(f"Gross PnL: {bt.summary['gross_pnl_sum']:.4f}")
print(f"Total Costs: {bt.summary['cost_sum']:.4f}")
print(f"Sharpe-like: {bt.summary['sharpe_like']:.3f}")
print(f"Max Drawdown: {bt.summary['max_drawdown']:.4f}")
print(f"\nTrades: {bt.trade_stats['n_trades']:.0f}")
print(f"Win Rate: {bt.trade_stats['win_rate']:.2%}")
print(f"Profit Factor: {bt.trade_stats['profit_factor']:.2f}")

# ---- training curves ----
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(hist["train_loss"], label="train")
axes[0, 0].plot(hist["val_loss"], ls="--", label="val")
axes[0, 0].legend(); axes[0, 0].set_title("Loss")

axes[0, 1].plot(hist["train_acc"], label="train")
axes[0, 1].plot(hist["val_acc"], ls="--", label="val")
axes[0, 1].legend(); axes[0, 1].set_title("Accuracy (%)")

axes[0, 2].plot(hist["aux_loss"], label="aux (load-balance)", c="orange")
axes[0, 2].legend(); axes[0, 2].set_title("Aux Loss")

axes[1, 0].plot(hist["val_pnl"], label="val_pnl", c="green")
axes[1, 0].plot(hist["val_score"], label="acc_wt_pnl score", c="purple")
axes[1, 0].legend(); axes[1, 0].set_title("PnL / Score")

bt.frame["cum_pnl"].plot(ax=axes[1, 1], c="black")
axes[1, 1].set_title(f"Backtest Cumulative PnL ({best_horizon})")

# ---- Expert routing analysis across validation set ----
model.eval()
all_weights = []
with torch.no_grad():
    for batch in va:
        if len(batch) == 7:
            xS, xL, xM, tS, tL, y, meta = batch
        else:
            xS, xL, xM, tS, tL, y = batch
        _, extras = model(
            x_short=xS.to(device), x_long=xL.to(device), x_market=xM.to(device),
            t_short=tS.to(device), t_long=tL.to(device),
            return_features=True,
        )
        all_weights.append(extras["weights"].cpu().numpy())

weights_arr = np.concatenate(all_weights, axis=0)  # [N_val, K]
n_experts = weights_arr.shape[1]

expert_labels = [f"Expert {i}" for i in range(n_experts)]
for i in range(n_experts):
    axes[1, 2].hist(weights_arr[:, i], bins=30, alpha=0.5, label=expert_labels[i])
axes[1, 2].legend()
axes[1, 2].set_title("Expert Routing Weight Distribution")

for ax in axes.ravel():
    ax.grid(True, alpha=0.3)
plt.suptitle(
    f"Regime-Aware MoE Final \u2014 {TICKER} {best_horizon} (Continuous Backtest)",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.savefig(
    RESULTS_PATH / f"{STUDY_NAME}_training_backtest.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

# ---- confusion matrix ----
pred_labels = logits.argmax(axis=1)
cm = confusion_matrix(ytrue, pred_labels, labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=ax,
    xticklabels=["Short", "Flat", "Long"],
    yticklabels=["Short", "Flat", "Long"],
)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix ({best_horizon} horizon)")
plt.tight_layout()
plt.savefig(
    RESULTS_PATH / f"{STUDY_NAME}_confusion.png",
    dpi=150, bbox_inches="tight",
)
plt.show()
print(classification_report(
    ytrue, pred_labels,
    target_names=["Short", "Flat", "Long"], digits=4,
))

# ---- expert routing summary ----
print(f"\n--- Expert Routing Summary (validation set) ---")
print(f"  Samples: {len(weights_arr)}")
for i in range(n_experts):
    w = weights_arr[:, i]
    dominant = (weights_arr.argmax(axis=1) == i).sum()
    print(f"  Expert {i}: mean={w.mean():.4f}  std={w.std():.4f}  "
          f"dominant={dominant} ({100*dominant/len(weights_arr):.1f}%)")

tracker = model.get_last_tracker()
print(f"\n  Router aux loss (last batch): {tracker.get('aux_loss', 0):.4f}")
print(f"  Dominant expert (last batch): {tracker.get('dominant_expert', '?')}")

# ---- save all artifacts ----
joblib.dump(study, RESULTS_PATH / f"{STUDY_NAME}_study.pkl")
tdf.to_csv(RESULTS_PATH / f"{STUDY_NAME}_trials.csv", index=False)

best_params_out = {
    **study.best_params,
    "best_value": float(study.best_value),
    "ticker": TICKER,
    "model": "RegimeAwareMoE",
    "best_horizon": best_horizon,
    "horizon_steps": HORIZONS[best_horizon],
    "all_horizons": HORIZONS,
    "horizon_thresholds": horizon_thresholds,
    "asset_feature_cols": FEAT,
    "market_feature_cols": MARKET_FEAT,
    "time_feature_cols": list(eval_base.time_feature_cols),
    "long_lookback": int(LONG_LOOKBACK),
    "short_len_range": list(SHORT_RANGE),
    "raster_bins": int(RBINS),
    "bar_rule": BAR_RULE,
    "session": f"USA CT {SESSION_START}-{SESSION_END}",
    "session_start": SESSION_START,
    "session_end": SESSION_END,
    "train_stride": TRAIN_STRIDE,
    "val_stride": VAL_STRIDE,
    "backtest_mode": "continuous",
    "transaction_cost_bps": TRANSACTION_COST_BPS,
    "slippage_bps": SLIPPAGE_BPS,
    "scoring": "pnl * (acc - baseline) if margin >= 2% else pnl * (acc / baseline)",
    "backtest_summary": bt.summary,
    "backtest_trade_stats": bt.trade_stats,
    "expert_routing_summary": {
        f"expert_{i}": {
            "mean_weight": float(weights_arr[:, i].mean()),
            "std_weight": float(weights_arr[:, i].std()),
            "dominant_pct": float((weights_arr.argmax(axis=1) == i).mean()),
        }
        for i in range(n_experts)
    },
}
with open(RESULTS_PATH / f"{STUDY_NAME}_best_params.json", "w") as f:
    json.dump(best_params_out, f, indent=2, default=float)

torch.save({
    "model_state_dict": model.state_dict(),
    "params": study.best_params,
    "history": hist,
    "asset_feature_cols": FEAT,
    "market_feature_cols": MARKET_FEAT,
    "time_feature_cols": list(eval_base.time_feature_cols),
    "horizon_thresholds": horizon_thresholds,
    "best_horizon": best_horizon,
    "all_horizons": HORIZONS,
}, RESULTS_PATH / f"{STUDY_NAME}_final_model.pth")

bt.frame.to_csv(RESULTS_PATH / f"{STUDY_NAME}_backtest_frame.csv")

print(f"\nsaved to {RESULTS_PATH}")
for pth in sorted(RESULTS_PATH.glob(f"{STUDY_NAME}*")):
    print(f"  - {pth.name}")